# FrodoBots-2K → sidewalk policy (Colab)

Trains the behavior-cloning policy on **real** FrodoBots-2K teleop: front camera frame → `(linear, angular)`.

**Runtime:** set **Runtime → Change runtime type → GPU** (T4 is fine; A100 faster).

The dataset ships as 24 zip *parts* (`output_rides_*.zip`, ~19–45 GB each). One part unzips to ~20–50 GB and holds many rides — that fits Colab disk. We download **one part**, use the **first N rides**, and train. (The full 769 GB Berkeley-7K set is a separate big-disk-VM job, not Colab.)

## 1. Install deps
Colab already has torch / torchvision / pandas; we just add OpenCV for video decode.

In [ ]:
!pip install -q opencv-python-headless
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Get the code
Clone the repo (push it to GitHub first — ask Aiden). Set `REPO_URL` to your remote.

In [ ]:
REPO_URL = 'https://github.com/Ace2932/earth-rover-challenge.git'  # <-- set this
import os
if not os.path.isdir('/content/earth-rover-challenge'):
    !git clone -q $REPO_URL /content/earth-rover-challenge
%cd /content/earth-rover-challenge
import sys; sys.path.insert(0, 'vision')
print('code ready')

## 3. Download one dataset part
`PART=0` is the smallest (~19 GB download, ~21 GB extracted). Bump the number for a different/larger part.

In [ ]:
import pandas as pd
PART = 0
parts = pd.read_csv('https://frodobots-2k-dataset.s3.ap-southeast-1.amazonaws.com/complete-dataset.csv')
url = parts.loc[PART, 'url']
print('part', PART, parts.loc[PART, 'compressed size'], '->', parts.loc[PART, 'uncompressed size'])
print(url)
!mkdir -p /content/frodo
!wget -q --show-progress -O /content/part.zip {url}
!unzip -q -o /content/part.zip -d /content/frodo && rm /content/part.zip
print('extracted')

## 4. Peek: how many rides did we get?

In [ ]:
from dataset import find_ride_dirs
rides = find_ride_dirs('/content/frodo')
print(len(rides), 'rides found; first few:')
for r in rides[:5]:
    print(' ', r)

## 5. Train
resnet18, first `--max-rides` rides. First epoch is slower (it decodes + caches frames as JPEGs); later epochs are fast. Bump `--epochs` / `--max-rides` for a stronger policy.

In [ ]:
!PYTHONPATH=vision python vision/train.py \
  --data /content/frodo \
  --backbone resnet18 --img 96 --stride 6 \
  --max-rides 20 --epochs 20 \
  --out vision/sidewalk_frodobots.pt

## 6. Download the trained policy
`sidewalk_frodobots.pt` plugs straight into `fuse.py` / the live loop (see `vision/README.md`).

In [ ]:
from google.colab import files
files.download('vision/sidewalk_frodobots.pt')

## Notes
- **Watch `steer_sign_acc`.** FrodoBots driving is mostly straight (angular ≈ 0), so a model can score low MSE by predicting ~0. More rides + higher `--stride` diversity help; a turn-weighted loss is the next improvement.
- **Bigger run:** raise `--max-rides` (or drop it to use every ride in the part), `--img 128`, more epochs.
- **Berkeley-7K** (the competition set) is 769 GB — needs a big-disk cloud VM, then subsample → back here to fine-tune.